# Relevancia por parte — ¿hay una configuración mejor del paseo?

| Decisión | Hoy | Lo que se prueba |
|---|---|---|
| Damping $d$: la probabilidad de seguir una flecha en vez de volver a empezar | 0.5 | 0.15, 0.3, 0.7, 0.85 |
| Dónde empieza el paseo | En las cláusulas donde la parte tiene un enunciado, todas por igual | Pesadas por cuántos enunciados tiene en cada una · solo donde tiene cargas (obligaciones y prohibiciones) · solo donde tiene derechos |
| Qué se lee como relevancia | Lo que la fila recibe de las cláusulas que la necesitan | Eso más el reinicio: el PageRank entero |
| Qué enlaces se caminan | `modifies`, `depends_on` y `references` | Cada uno solo · sin `references` · los tres más los términos definidos |
| Enlaces repetidos entre dos cláusulas | Cuentan cada vez | Cuentan una vez |
| Cláusulas sin salida | Su peso vuelve al inicio | Su peso se reparte entre todas las filas |

No se prueba dar un peso distinto a cada tipo de enlace: no hay con qué justificar qué enlace
pesa más.

| Término | Significado |
|---|---|
| lectura | Un contrato leído desde una de sus dos partes (las dos con más enunciados). Nueve contratos y dos partes: 18 lecturas. |
| camino de la parte | Las cláusulas a las que el paseo llega saliendo de las de la parte. |
| fila con puesto | Una fila que recibe algo: la necesita alguna cláusula del camino de la parte. Las demás quedan al final, en el orden del contrato («–» en la tabla de la app). |
| empate | Una fila con puesto que vale lo mismo que otra. |
| pareja mutua | Dos cláusulas que se necesitan la una a la otra. |
| top 5 | Las cinco primeras filas. |

El damping ya se miró valor a valor, para el contrato entero, en
[`relevance_damping.ipynb`](relevance_damping.ipynb). Aquí se repite con cinco valores y leído
desde cada parte.

Se ejecuta con el entorno de `server/` (usa `scipy`, que ya está en sus dependencias).

In [1]:
import itertools
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

from scipy.stats import kendalltau, spearmanr

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'infra').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'server'))
from services.graph.knowledge import personalized_pagerank as ppr

# La configuración de hoy y lo que se prueba en cada decisión.
TODAY = {
    'damping': ppr.RELEVANCE_DAMPING, 'inicio': 'por igual', 'lectura': 'lo recibido',
    'enlaces': 'los tres', 'repetidos': 'cada vez', 'sin salida': 'al inicio',
}
OPTIONS = {
    'damping': [0.15, 0.3, 0.5, 0.7, 0.85],
    'inicio': ['por igual', 'por enunciados', 'solo cargas', 'solo derechos'],
    'lectura': ['lo recibido', 'con el reinicio'],
    'enlaces': ['los tres', 'solo modifies', 'solo depends_on', 'solo references', 'sin references', 'con términos'],
    'repetidos': ['cada vez', 'una vez'],
    'sin salida': ['al inicio', 'a todas las filas'],
}
# Las relaciones del módulo que camina cada juego de enlaces. `uses_term` solo existe aquí.
RELATIONS = {
    'los tres': {'modified_by', 'depends_on', 'refers_to'},
    'solo modifies': {'modified_by'},
    'solo depends_on': {'depends_on'},
    'solo references': {'refers_to'},
    'sin references': {'modified_by', 'depends_on'},
    'con términos': {'modified_by', 'depends_on', 'refers_to', 'uses_term'},
}
NAMES = {
    'BELLICUM': 'Bellicum', 'CARTAO': 'Santander', 'CONTRATO_AREQUIPA': 'Arequipa', 'SteelVault': 'SteelVault',
    'Synthetic_Halverson': 'HalversonCorvin', 'Synthetic_Orvane': 'OrvaneTalvera',
}
# Los tres contratos con suficientes enlaces para que el orden tenga algo que mover.
BIG = ('Bellicum', 'Santander', 'OrvaneTalvera')
ZERO = 1e-15
WHO = 41


def term_links(kg):
    """Una cláusula que usa una palabra definida necesita la cláusula que la define: una flecha
    por palabra, como el «Uses terms from» de la tabla. Da (quién lee, a quién necesita, palabra)."""
    clause_ids = {clause['id'] for clause in kg['clauses']}
    home = {s['id']: s.get('clauseId') for kind in ppr.DEONTIC_COLLECTIONS for s in kg.get(kind) or []}
    defined_in = {}
    for edge in kg['edges']:
        if edge['type'] == 'defines' and edge['source'] in clause_ids:
            defined_in.setdefault(edge['target'], edge['source'])
    # Solo las palabras que la tabla enseña: con alguna letra y con definición.
    words = {term['id']: term['term'] for term in kg.get('definedTerms') or []
             if re.search(r'[^\W\d_]', term.get('term') or '') and (term.get('definition') or '').strip()}
    seen, out = set(), []
    for edge in kg['edges']:
        if edge['type'] != 'uses' or edge['target'] not in words:
            continue
        reader = edge['source'] if edge['source'] in clause_ids else home.get(edge['source'])
        needed = defined_in.get(edge['target'])
        if reader and needed and reader != needed and (reader, edge['target']) not in seen:
            seen.add((reader, edge['target']))
            out.append((reader, needed, words[edge['target']]))
    return out


class Contract:
    def __init__(self, path):
        stem = path.stem.removeprefix('root_')
        self.name = next((short for prefix, short in NAMES.items() if stem.startswith(prefix)), stem)
        self.kg = json.loads(path.read_text(encoding='utf-8'))
        self.ids = [clause['id'] for clause in self.kg['clauses']]
        self.index = {clause: at for at, clause in enumerate(self.ids)}
        self.ref = {clause['id']: clause.get('ref') or clause['id'] for clause in self.kg['clauses']}
        # Los enunciados de cada parte: las obligaciones y prohibiciones que carga y los derechos que tiene.
        held = defaultdict(list)
        for kind in ppr.DEONTIC_COLLECTIONS:
            for statement in self.kg.get(kind) or []:
                party = statement.get('benefitPartyId' if kind == 'rights' else 'burdenPartyId')
                if party and statement.get('clauseId') in self.index:
                    held[party].append((kind, statement['clauseId'], statement['id']))
        # El par: las dos partes con más enunciados. Las filas: las cláusulas con un enunciado de alguna.
        self.pair = sorted(held, key=lambda party: -len(held[party]))[:2]
        self.held = {party: held[party] for party in self.pair}
        names = {party['id']: party['name'] for party in self.kg['parties']}
        self.party = {party: names[party] for party in self.pair}
        # El tamaño de la fila: cuántos enunciados del par tiene.
        self.size = Counter(clause for party in self.pair for _, clause, _ in self.held[party])
        self.rows = [clause for clause in self.ids if clause in self.size]
        terms = term_links(self.kg)
        self.links = [(link.reader, link.needed, link.relation) for link in ppr.needs(self.kg)]
        self.links += [(reader, needed, 'uses_term') for reader, needed, _ in terms]
        # Por cada cláusula que define palabras: cuántas cláusulas usan cada una.
        self.words = defaultdict(Counter)
        for _, needed, word in terms:
            self.words[needed][word] += 1

    def label(self, clause):
        return '§' + self.ref[clause]

    def who(self, party):
        name = self.party[party]
        return f"{self.name} · {name if len(name) <= 20 else name[:19] + '…'}"


contracts = [Contract(path) for path in sorted((ROOT / 'infra' / 'json' / 'kg').glob('root_*.json'))]
big = [c for c in contracts if c.name in BIG]

print(f"{'contrato':19} {'filas':>5} {'flechas':>8} {'repetidas':>10} {'por términos':>13}")
for c in contracts:
    arrows = [(reader, needed) for reader, needed, relation in c.links if relation != 'uses_term']
    print(f'{c.name:19} {len(c.rows):>5} {len(arrows):>8} {len(arrows) - len(set(arrows)):>10} {len(c.links) - len(arrows):>13}')

print('\nLas dos partes de cada contrato y cuántos enunciados tiene cada una')
for c in contracts:
    print(f'{c.name:19} ' + ' · '.join(f'{c.party[party]} ({len(c.held[party])})' for party in c.pair))

contrato            filas  flechas  repetidas  por términos
Bellicum               91      273         85           520
Santander              81       83         27            28
Arequipa               14        3          0             0
SteelVault             22        6          2             0
HalversonCorvin        18        7          2             1
OrvaneTalvera          48       29          1            45
lease_contract_10      20        8          1             0
lease_contract_166      9        0          0             0
lease_contract_201      8        7          3             0

Las dos partes de cada contrato y cuántos enunciados tiene cada una
Bellicum            Miltenyi Biotec GmbH (311) · Bellicum Pharmaceuticals, Inc. (299)
Santander           Titular do cartão (120) · BANCO SANTANDER (BRASIL) S.A. (77)
Arequipa            Alicia Marybel Diaz Zea (15) · Arequipa Todo Servicios (6)
SteelVault          National Credit Report.com, LLC (41) · Equidata, Inc. (37)
Halver

## 1. El paseo, con sus seis decisiones abiertas

`reading` es el paseo del módulo del servidor con las seis decisiones como parámetros. Con la
configuración de hoy tiene que dar lo mismo que `ppr.compute`: la celda lo comprueba en las 18
lecturas antes de probar nada.

In [ ]:
def reading(contract, party, cfg):
    index, rows, n = contract.index, contract.rows, len(contract.ids)
    arrows = [(reader, needed) for reader, needed, relation in contract.links if relation in RELATIONS[cfg['enlaces']]]
    if cfg['repetidos'] == 'una vez':
        arrows = list(dict.fromkeys(arrows))
    successors = [[] for _ in range(n)]
    for reader, needed in arrows:
        successors[index[reader]].append(index[needed])

    # Dónde empieza: las cláusulas con un enunciado de la parte, o solo con los de un tipo.
    begins = Counter()
    for kind, clause, _ in contract.held[party]:
        if cfg['inicio'] == 'solo cargas' and kind == 'rights':
            continue
        if cfg['inicio'] == 'solo derechos' and kind != 'rights':
            continue
        begins[clause] += 1
    if not begins:
        return None
    if cfg['inicio'] != 'por enunciados':
        begins = Counter(dict.fromkeys(begins, 1))
    prior = [0.0] * n
    for clause, weight in begins.items():
        prior[index[clause]] = weight / begins.total()
    # A dónde va el peso de una cláusula sin salida.
    refill = prior
    if cfg['sin salida'] == 'a todas las filas':
        refill = [0.0] * n
        for clause in rows:
            refill[index[clause]] = 1 / len(rows)

    d, rank = cfg['damping'], prior[:]
    for _ in range(ppr.MAX_ITERATIONS):
        nxt, dangling = [0.0] * n, 0.0
        for j, targets in enumerate(successors):
            if not targets:
                dangling += rank[j]
                continue
            share = d * rank[j] / len(targets)
            for target in targets:
                nxt[target] += share
        delta = 0.0
        for i in range(n):
            nxt[i] += (1 - d) * prior[i] + d * dangling * refill[i]
            delta += abs(nxt[i] - rank[i])
        rank = nxt
        if delta < ppr.TOLERANCE:
            break

    # Lo que cada cláusula recibe de las que la necesitan, y cuántas del camino de la parte son.
    received, readers = [0.0] * n, defaultdict(set)
    for j, targets in enumerate(successors):
        for target in targets:
            received[target] += d * rank[j] / len(targets)
            if rank[j] > 0:
                readers[target].add(j)
    linked = set(arrows)
    mutual = {needed for reader, needed in arrows if (needed, reader) in linked}

    got = {row: received[index[row]] for row in rows}
    score = got if cfg['lectura'] == 'lo recibido' else {row: rank[index[row]] for row in rows}
    order = sorted(rows, key=lambda row: (-score[row], index[row]))
    return SimpleNamespace(
        score=score, received=got, order=order, starts=len(begins),
        # Las filas con puesto, y las que tienen algún valor con lo que se lee (las mismas, sin el reinicio).
        reasoned=[row for row in order if got[row] > ZERO],
        placed=[row for row in order if score[row] > ZERO],
        needed_by={row: len(readers[index[row]]) for row in rows},
        mutual={row for row in rows if row in mutual},
    )


for c in contracts:
    counted = [statement for party in c.pair for _, _, statement in c.held[party]]
    for party in c.pair:
        module = ppr.compute(c.kg, counted, [statement for _, _, statement in c.held[party]]).by_clause
        here = reading(c, party, TODAY).received
        assert all(abs(module[row] - here[row]) < 1e-9 for row in c.rows), c.who(party)
print(f'Con la configuración de hoy, el paseo de este cuaderno da lo mismo que el módulo en las {2 * len(contracts)} lecturas.')

Con la configuración de hoy, el paseo de este cuaderno da lo mismo que el módulo en las 18 lecturas.


## 2. La configuración de hoy

El punto de partida. Para cada lectura: cuántas filas tienen puesto, cuántos empates hay entre
ellas y cuáles son las cinco primeras. Debajo de cada contrato, cuántas de las cinco primeras
son las mismas cláusulas para las dos partes (o de las que haya, si tienen puesto menos de cinco).

In [3]:
def ties(res):
    """Filas con puesto que valen lo mismo que otra."""
    values = [round(res.received[row], 12) for row in res.reasoned]
    return len(values) - len(set(values))


def buried(res):
    """Filas que alguna cláusula necesita y que quedan por debajo de una fila que nadie necesita."""
    first_without = next((at for at, row in enumerate(res.order) if res.received[row] <= ZERO), len(res.order))
    return sum(1 for row in res.order[first_without:] if res.received[row] > ZERO)


def weak_pair(res, row):
    """Una fila de una pareja mutua que necesitan dos cláusulas como mucho."""
    return row in res.mutual and res.needed_by[row] <= 2


def below(res, at):
    """Cuántas cláusulas que necesitan cinco o más quedan por debajo de ese puesto."""
    return sum(1 for row in res.placed[at + 1:] if res.needed_by[row] >= 5)


def overtaken(res):
    """Lo anterior, para cada fila de una pareja mutua que está entre las cinco primeras."""
    return sum(below(res, at) for at, row in enumerate(res.placed[:5]) if weak_pair(res, row))


def first_share(res):
    """La parte de todo lo recibido que se lleva la fila que más recibe."""
    total = sum(res.received.values())
    return max(res.received.values()) / total if total else 0.0


def values(contract, res):
    return [round(res.score[row], 12) for row in contract.rows]


def like_size(contract, res):
    """ρ de Spearman entre el valor de la fila y su tamaño. None si no hay nada que comparar."""
    xs, ys = values(contract, res), [contract.size[row] for row in contract.rows]
    return spearmanr(xs, ys).statistic if len(set(xs)) > 1 and len(set(ys)) > 1 else None


def alike(contract, a, b):
    """τ de Kendall entre los órdenes de las dos partes: 1 es el mismo orden."""
    xs, ys = values(contract, a), values(contract, b)
    return kendalltau(xs, ys).statistic if len(set(xs)) > 1 and len(set(ys)) > 1 else None


def shared(a, b):
    return len(set(a.placed[:5]) & set(b.placed[:5]))


today = {c.name: [reading(c, party, TODAY) for party in c.pair] for c in contracts}

print(f"{'lectura':{WHO}} {'filas':>5} {'con puesto':>10} {'empates':>8}   las cinco primeras")
for c in contracts:
    for party, res in zip(c.pair, today[c.name]):
        print(f'{c.who(party):{WHO}} {len(c.rows):>5} {len(res.reasoned):>10} {ties(res):>8}   ' + ' '.join(c.label(row) for row in res.placed[:5]))
    a, b = today[c.name]
    most = min(5, max(len(a.placed), len(b.placed)))
    print(f"{'':{WHO}} " + (f'en común: {shared(a, b)} de {most}' if most else 'ninguna fila tiene puesto') + '\n')
print(f'En total: {sum(len(res.reasoned) for pair in today.values() for res in pair)} filas con puesto y '
      f'{sum(ties(res) for pair in today.values() for res in pair)} empates.')

lectura                                   filas con puesto  empates   las cinco primeras
Bellicum · Miltenyi Biotec GmbH              91         58        5   §4.10 §3.2 §6.5 §7.7 §14.3
Bellicum · Bellicum Pharmaceut…              91         59        4   §4.10 §3.2 §7.7 §6.5 §14.3
                                          en común: 5 de 5

Santander · Titular do cartão                81         22        8   §18.8 §5.4 §6.1 §6.3 §8.1
Santander · BANCO SANTANDER (BR…             81         16        3   §18.8 §6.1 §6.3 §8.1 §13.1
                                          en común: 4 de 5

Arequipa · Alicia Marybel Diaz…              14          3        2   §10 §11 §14
Arequipa · Arequipa Todo Servi…              14          3        2   §10 §11 §14
                                          en común: 3 de 3

SteelVault · National Credit Rep…            22          1        0   §11
SteelVault · Equidata, Inc.                  22          1        0   §11
                                

## 3. Un parámetro cada vez

Cada alternativa, con el resto como hoy. Las columnas suman las 18 lecturas, salvo las del top 5,
que miran solo Bellicum, Santander y OrvaneTalvera: los tres contratos con suficientes enlaces
para que el orden tenga algo que mover (6 lecturas, 30 puestos).

| Columna | Qué cuenta |
|---|---|
| con puesto | Filas con puesto. Menos que hoy: hay filas que pierden su motivo. |
| empates | Filas con puesto que valen lo mismo que otra. |
| sin inicio | Lecturas en las que la parte no tiene ninguna cláusula donde empezar. |
| enterradas | Filas que alguna cláusula necesita y que quedan por debajo de una fila que nadie necesita. |
| adelantadas | Cláusulas que necesitan cinco o más y que quedan por debajo de una fila de una pareja mutua que está entre las cinco primeras y a la que necesitan dos como mucho. |
| top 5 = hoy | De las cinco primeras de hoy, cuántas siguen entre las cinco primeras (de 30). |
| top 5 común | Cuántas de las cinco primeras comparten las dos partes (de 15). Más alto: las partes se distinguen menos. |
| 1.ª Bellicum | La parte de todo lo recibido que se lleva la primera fila de Bellicum (la mayor de sus dos lecturas). |
| ρ tamaño | El parecido medio entre el orden y el tamaño de la fila (ρ de Spearman, sin signo). Más alto: el orden repite «by size». |

In [4]:
def summary(cfg):
    """Una configuración en las 18 lecturas."""
    s = SimpleNamespace(cfg=cfg, no_start=0, placed=0, ties=0, buried=0, overtaken=0, top5_kept=0, top5_shared=0, bellicum_first=0.0)
    sizes = []
    for c in contracts:
        results = [reading(c, party, cfg) for party in c.pair]
        for res, base in zip(results, today[c.name]):
            if res is None:
                s.no_start += 1
                continue
            s.placed += len(res.reasoned)
            s.ties += ties(res)
            s.buried += buried(res)
            s.overtaken += overtaken(res)
            rho = like_size(c, res)
            if rho is not None:
                sizes.append(abs(rho))
            if c.name in BIG:
                s.top5_kept += len(set(res.placed[:5]) & set(base.placed[:5]))
            if c.name == 'Bellicum':
                s.bellicum_first = max(s.bellicum_first, first_share(res))
        if c.name in BIG and all(results):
            s.top5_shared += shared(*results)
    s.like_size = sum(sizes) / len(sizes)
    return s


def columns(s):
    return (f'{s.placed:>10} {s.ties:>8} {s.no_start:>10} {s.buried:>10} {s.overtaken:>11} {s.top5_kept:>11} {s.top5_shared:>11} '
            f'{s.bellicum_first:>12.0%} {s.like_size:>9.2f}')


HEAD = (f"{'con puesto':>10} {'empates':>8} {'sin inicio':>10} {'enterradas':>10} {'adelantadas':>11} {'top 5 = hoy':>11} {'top 5 común':>11} "
        f"{'1.ª Bellicum':>12} {'ρ tamaño':>9}")
alone = {}
print(f"{'':26}" + HEAD)
for key, options in OPTIONS.items():
    print(key)
    for value in options:
        alone[key, value] = summary({**TODAY, key: value})
        print(f"  {str(value) + (' (hoy)' if TODAY[key] == value else ''):24}" + columns(alone[key, value]))

                          con puesto  empates sin inicio enterradas adelantadas top 5 = hoy top 5 común 1.ª Bellicum  ρ tamaño
damping
  0.15                           221       43          0          0           0          25           7          11%      0.20
  0.3                            221       43          0          0           0          27           8          12%      0.20
  0.5 (hoy)                      221       43          0          0           0          30          10          13%      0.20
  0.7                            221       43          0          0           9          28           9          15%      0.20
  0.85                           221       43          0          0          44          22           6          18%      0.20
inicio
  por igual (hoy)                221       43          0          0           0          30          10          13%      0.20
  por enunciados                 221       41          0          0           0          25     

## 4. El damping y las parejas mutuas

Qué filas tienen puesto no depende de $d$: lo decide el grafo. Lo que cambia es el orden entre
ellas, y cambia por las parejas mutuas: el paseo va y vuelve entre las dos cláusulas, y cuanto
más alto es $d$ más rondas aguanta sin volver a empezar.

- La primera tabla da el puesto de la primera fila de una pareja mutua a la que necesitan dos
  cláusulas como mucho, y cuántas cláusulas que necesitan cinco o más quedan por debajo de ella.
- La segunda, qué parte de todo lo recibido queda en filas de parejas mutuas.
- La tercera, las siete primeras filas de Bellicum con cada $d$.

In [5]:
DAMPINGS = OPTIONS['damping']
by_damping = {(c.name, party, d): reading(c, party, {**TODAY, 'damping': d}) for c in big for party in c.pair for d in DAMPINGS}

print('Puesto de la primera fila de una pareja mutua a la que necesitan dos como mucho.')
print('Entre paréntesis, cuántas cláusulas que necesitan 5 o más quedan debajo.')
print(f"{'':{WHO}}" + ''.join(f"{f'd = {d}':>17}" for d in DAMPINGS))
for c in big:
    for party in c.pair:
        cells = []
        for d in DAMPINGS:
            res = by_damping[c.name, party, d]
            at = next((at for at, row in enumerate(res.placed) if weak_pair(res, row)), None)
            cells.append('—' if at is None else f'{at + 1}.º {c.label(res.placed[at])} ({below(res, at)})')
        print(f'{c.who(party):{WHO}}' + ''.join(f'{cell:>17}' for cell in cells))

print('\nParte de todo lo recibido que queda en filas de parejas mutuas')
print(f"{'':{WHO}}" + ''.join(f"{f'd = {d}':>10}" for d in DAMPINGS))
for c in big:
    for party in c.pair:
        shares = [sum(res.received[row] for row in res.mutual) / sum(res.received.values()) for res in (by_damping[c.name, party, d] for d in DAMPINGS)]
        print(f'{c.who(party):{WHO}}' + ''.join(f'{share:>10.0%}' for share in shares))

bellicum = next(c for c in contracts if c.name == 'Bellicum')
print('\nBellicum: las siete primeras filas.')
print('Entre paréntesis, cuántas cláusulas la necesitan; * si es de una pareja mutua y la necesitan dos como mucho.')
for party in bellicum.pair:
    print(bellicum.party[party])
    for d in DAMPINGS:
        res = by_damping['Bellicum', party, d]
        print(f"  {f'd = {d}':10}" + '  '.join(f"{bellicum.label(row)} ({res.needed_by[row]}{'*' if weak_pair(res, row) else ''})" for row in res.placed[:7]))

Puesto de la primera fila de una pareja mutua a la que necesitan dos como mucho.
Entre paréntesis, cuántas cláusulas que necesitan 5 o más quedan debajo.
                                                  d = 0.15          d = 0.3          d = 0.5          d = 0.7         d = 0.85
Bellicum · Miltenyi Biotec GmbH              12.º §9.1 (4)     8.º §9.1 (6)     8.º §9.1 (6)     8.º §9.1 (6)     6.º §9.1 (8)
Bellicum · Bellicum Pharmaceut…               7.º §9.1 (8)     6.º §9.1 (8)     6.º §9.1 (8)    5.º §15.2 (9)    2.º §9.1 (11)
Santander · Titular do cartão               10.º §15.1 (0)   10.º §15.1 (0)   10.º §15.1 (0)   10.º §15.1 (0)   10.º §15.1 (0)
Santander · BANCO SANTANDER (BR…            12.º §15.2 (0)   12.º §15.2 (0)   12.º §15.2 (0)   12.º §15.2 (0)   12.º §15.2 (0)
OrvaneTalvera · Orvane Bioprocess G…         1.º §11.2 (0)    1.º §11.2 (0)    1.º §11.2 (0)     1.º §9.1 (0)     1.º §9.1 (0)
OrvaneTalvera · Talvera Therapeutic…          1.º §2.2 (0)     1.º §2.2 (0)     1.º 

## 5. Dónde empieza el paseo

Filas con puesto con cada inicio y, entre paréntesis, en cuántas cláusulas empieza el paseo.
«—»: la parte no tiene ningún enunciado de ese tipo y el paseo no puede empezar.

In [6]:
STARTS = OPTIONS['inicio']
print(f"{'':{WHO}}" + ''.join(f'{start:>16}' for start in STARTS))
for c in contracts:
    for party in c.pair:
        cells = []
        for start in STARTS:
            res = reading(c, party, {**TODAY, 'inicio': start})
            cells.append('—' if res is None else f'{len(res.reasoned)} ({res.starts})')
        print(f'{c.who(party):{WHO}}' + ''.join(f'{cell:>16}' for cell in cells))

                                                por igual  por enunciados     solo cargas   solo derechos
Bellicum · Miltenyi Biotec GmbH                   58 (80)         58 (80)         58 (63)         52 (52)
Bellicum · Bellicum Pharmaceut…                   59 (79)         59 (79)         59 (66)         56 (49)
Santander · Titular do cartão                     22 (52)         22 (52)         22 (36)         10 (25)
Santander · BANCO SANTANDER (BR…                  16 (55)         16 (55)         14 (26)          8 (36)
Arequipa · Alicia Marybel Diaz…                    3 (12)          3 (12)          0 (11)           3 (2)
Arequipa · Arequipa Todo Servi…                     3 (4)           3 (4)               —           3 (4)
SteelVault · National Credit Rep…                  1 (19)          1 (19)          1 (19)           0 (3)
SteelVault · Equidata, Inc.                        1 (18)          1 (18)          1 (14)           0 (8)
HalversonCorvin · Corvin Outreach Gro…        

## 6. Qué se lee como relevancia

Hoy se lee lo que la fila recibe. Con el reinicio, una fila de la parte vale algo solo por ser
suya, aunque ninguna cláusula la necesite, y puede quedar por encima de filas que sí tienen
motivo. La tabla cuenta esas filas «enterradas» y compara el parecido del orden con el tamaño de
la fila (ρ de Spearman: 1 sería el mismo orden que «by size»).

In [7]:
def signed(value):
    return '—' if value is None else f'{value:+.2f}'


print(f"{'':{WHO}} {'enterradas':>10} {'ρ tamaño: lo recibido':>22} {'con el reinicio':>16}")
for c in contracts:
    for party, base in zip(c.pair, today[c.name]):
        res = reading(c, party, {**TODAY, 'lectura': 'con el reinicio'})
        print(f'{c.who(party):{WHO}} {buried(res):>10} {signed(like_size(c, base)):>22} {signed(like_size(c, res)):>16}')

                                          enterradas  ρ tamaño: lo recibido  con el reinicio
Bellicum · Miltenyi Biotec GmbH                    6                  -0.03            +0.18
Bellicum · Bellicum Pharmaceut…                    7                  -0.06            +0.07
Santander · Titular do cartão                      4                  +0.07            +0.26
Santander · BANCO SANTANDER (BR…                   4                  -0.02            +0.12
Arequipa · Alicia Marybel Diaz…                    2                  +0.23            +0.10
Arequipa · Arequipa Todo Servi…                    0                  +0.23            +0.43
SteelVault · National Credit Rep…                  0                  +0.24            +0.25
SteelVault · Equidata, Inc.                        0                  +0.24            +0.51
HalversonCorvin · Corvin Outreach Gro…             3                  -0.30            +0.25
HalversonCorvin · Halverson Data Syst…             0                  

## 7. Qué enlaces se caminan

Filas con puesto con cada juego de enlaces.

In [8]:
LINKS = OPTIONS['enlaces']
print(f"{'':{WHO}}" + ''.join(f'{name:>16}' for name in LINKS))
for c in contracts:
    for party in c.pair:
        print(f'{c.who(party):{WHO}}' + ''.join(f"{len(reading(c, party, {**TODAY, 'enlaces': name}).reasoned):>16}" for name in LINKS))

                                                 los tres   solo modifies solo depends_on solo references  sin references    con términos
Bellicum · Miltenyi Biotec GmbH                        58              24              31              45              46              64
Bellicum · Bellicum Pharmaceut…                        59              24              29              46              46              65
Santander · Titular do cartão                          22              12              11               0              22              28
Santander · BANCO SANTANDER (BR…                       16              13               3               0              16              20
Arequipa · Alicia Marybel Diaz…                         3               3               0               0               3               3
Arequipa · Arequipa Todo Servi…                         3               3               0               0               3               3
SteelVault · National Credit Rep… 

Con los términos definidos, una cláusula que usa una palabra necesita a la cláusula que la
define. La tabla enseña qué fila queda primera entonces, qué parte de todo lo recibido se lleva
(contando cada enlace, y contando una vez los repetidos), qué puesto tiene hoy y por qué palabras
sube: las tres que más cláusulas usan, de todas las que define.

In [9]:
print(f"{'':{WHO}} {'primera':>8} {'se lleva':>9} {'una vez':>8} {'hoy':>6}   las palabras que más cláusulas usan (cuántas)")
for c in big:
    for party, base in zip(c.pair, today[c.name]):
        res = reading(c, party, {**TODAY, 'enlaces': 'con términos'})
        once = reading(c, party, {**TODAY, 'enlaces': 'con términos', 'repetidos': 'una vez'})
        first = res.placed[0]
        was = f'{base.reasoned.index(first) + 1}.ª' if first in base.reasoned else '–'
        words = c.words[first]
        print(f'{c.who(party):{WHO}} {c.label(first):>8} {res.received[first] / sum(res.received.values()):>9.0%} '
              f'{once.received[first] / sum(once.received.values()):>8.0%} {was:>6}   '
              + ', '.join(f'«{word}» ({uses})' for word, uses in words.most_common(3)) + f" · define {len(words)} palabra{'s' if len(words) > 1 else ''}")

                                           primera  se lleva  una vez    hoy   las palabras que más cláusulas usan (cuántas)
Bellicum · Miltenyi Biotec GmbH               §1.1       50%      37%   24.ª   «Miltenyi Products» (46), «Miltenyi Product» (38), «Bellicum Product» (29) · define 53 palabras
Bellicum · Bellicum Pharmaceut…               §1.1       51%      38%   21.ª   «Miltenyi Products» (46), «Miltenyi Product» (38), «Bellicum Product» (29) · define 53 palabras
Santander · Titular do cartão                 §5.8       18%      18%      –   «IOF» (10) · define 1 palabra
Santander · BANCO SANTANDER (BR…              §5.8       22%      22%      –   «IOF» (10) · define 1 palabra
OrvaneTalvera · Orvane Bioprocess G…         §13.1       12%      12%      –   «Term» (3) · define 1 palabra
OrvaneTalvera · Talvera Therapeutic…         §13.1       25%      26%      –   «Term» (3) · define 1 palabra


## 8. Enlaces repetidos y cláusulas sin salida

**Repetidos.** Cuando dos enlaces dicen lo mismo entre dos cláusulas, hoy son dos caminos.
Contarlos una vez no cambia qué filas tienen puesto; la tabla enseña las lecturas donde cambian
los empates.

**Sin salida.** Hoy, el peso que llega a una cláusula que no necesita a ninguna vuelve a las
cláusulas de la parte. Repartirlo entre todas las filas lo lleva también a cláusulas que no están
en el camino de la parte. La segunda tabla enseña las filas que ganan puesto así, cuántas de
ellas necesita alguna cláusula del camino de la parte, y cuánto se parecen los órdenes de las dos
partes (τ de Kendall: 1 es el mismo orden).

In [10]:
print('Enlaces repetidos: empates entre las filas con puesto, donde cambian')
print(f"{'':{WHO}} {'cada vez':>9} {'una vez':>8}")
for c in contracts:
    for party, base in zip(c.pair, today[c.name]):
        once = reading(c, party, {**TODAY, 'repetidos': 'una vez'})
        assert set(once.reasoned) == set(base.reasoned)
        if ties(once) != ties(base):
            print(f'{c.who(party):{WHO}} {ties(base):>9} {ties(once):>8}')

print('\nSin salida: filas que ganan puesto si el peso se reparte entre todas las filas')
gained = needed = 0
for c in contracts:
    spread = [reading(c, party, {**TODAY, 'sin salida': 'a todas las filas'}) for party in c.pair]
    for party, base, res in zip(c.pair, today[c.name], spread):
        new = [row for row in res.reasoned if row not in base.reasoned]
        gained += len(new)
        needed += sum(1 for row in new if base.needed_by[row] > 0)
        if new:
            print(f'{c.who(party):{WHO}} ' + ' '.join(c.label(row) for row in new))
print(f'{gained} filas ganan puesto; a {needed} de ellas las necesita alguna cláusula del camino de la parte.')

print('\nParecido entre los órdenes de las dos partes (τ)')
print(f"{'':19} {'al inicio':>10} {'a todas las filas':>18}")
for c in contracts:
    spread = [reading(c, party, {**TODAY, 'sin salida': 'a todas las filas'}) for party in c.pair]
    before, after = alike(c, *today[c.name]), alike(c, *spread)
    if before is not None and after is not None:
        print(f'{c.name:19} {before:>10.2f} {after:>18.2f}')

Enlaces repetidos: empates entre las filas con puesto, donde cambian
                                           cada vez  una vez
Bellicum · Miltenyi Biotec GmbH                   5        7
Bellicum · Bellicum Pharmaceut…                   4        6
Santander · BANCO SANTANDER (BR…                  3        5
lease_contract_10 · Zhonghai Trust Co.,…          0        1
lease_contract_10 · Shaanxi Tianren Org…          0        1
lease_contract_201 · Bozhou Fengyi Chine…         0        1

Sin salida: filas que ganan puesto si el peso se reparte entre todas las filas
Bellicum · Miltenyi Biotec GmbH           §9.3
Santander · Titular do cartão             §3.4
Santander · BANCO SANTANDER (BR…          §11 §11.1 §11.2 §11.3 §12 §12.2 §12.3
OrvaneTalvera · Orvane Bioprocess G…      §5.2
OrvaneTalvera · Talvera Therapeutic…      §9.1 §9.3
lease_contract_201 · Anhui Xuelingxian P… §2
13 filas ganan puesto; a 0 de ellas las necesita alguna cláusula del camino de la parte.

Parecido entre l

## 9. Las 960 combinaciones

Un parámetro que solo no mejora podría mejorar junto a otro. Esta celda calcula todas las
combinaciones y les pasa seis pruebas. Las pruebas son criterios de este experimento, pensados
para descartar los defectos vistos en los apartados anteriores; ninguna viene de un criterio
externo.

In [11]:
grid = [summary(dict(zip(OPTIONS, combo))) for combo in itertools.product(*OPTIONS.values())]
base = alone['damping', TODAY['damping']]
TESTS = {
    'todas las lecturas tienen dónde empezar': lambda s: s.no_start == 0,
    'ninguna fila con motivo queda debajo de una sin motivo': lambda s: s.buried == 0,
    'ninguna pareja mutua adelanta a cláusulas que necesitan 5 o más': lambda s: s.overtaken == 0,
    'la primera fila de Bellicum no se lleva más de un tercio': lambda s: s.bellicum_first <= 1 / 3,
    'no hay menos filas con puesto que hoy': lambda s: s.placed >= base.placed,
    'las partes no comparten más top 5 que hoy': lambda s: s.top5_shared <= base.top5_shared,
}
print(f'{len(grid)} configuraciones\n')
for name, test in TESTS.items():
    print(f'{sum(test(s) for s in grid):>4} cumplen: {name}')
passing = [s for s in grid if all(test(s) for test in TESTS.values())]
print(f'{len(passing):>4} cumplen las seis\n')

print('De las que cumplen las seis, cuántas tienen cada valor')
for key, options in OPTIONS.items():
    print(f'  {key:11}' + ' · '.join(f'{value}: {sum(s.cfg[key] == value for s in passing)}' for value in options))

print('\nQué pruebas falla cada alternativa, con el resto como hoy')
for (key, value), s in alone.items():
    if TODAY[key] != value:
        print(f"  {f'{key}: {value}':30}" + ('; '.join(name for name, test in TESTS.items() if not test(s)) or 'ninguna'))


def changes(cfg):
    return ', '.join(f'{key}: {value}' for key, value in cfg.items() if TODAY[key] != value) or 'la de hoy'


print('\nLas que cumplen las seis, con menos empates primero')
print(f"{'con puesto':>10} {'empates':>8} {'top 5 = hoy':>11} {'top 5 común':>11}   qué cambia respecto a hoy")
for s in sorted(passing, key=lambda s: (s.ties, -s.top5_kept)):
    print(f'{s.placed:>10} {s.ties:>8} {s.top5_kept:>11} {s.top5_shared:>11}   {changes(s.cfg)}')

960 configuraciones

 720 cumplen: todas las lecturas tienen dónde empezar
 480 cumplen: ninguna fila con motivo queda debajo de una sin motivo
 684 cumplen: ninguna pareja mutua adelanta a cláusulas que necesitan 5 o más
 804 cumplen: la primera fila de Bellicum no se lleva más de un tercio
 260 cumplen: no hay menos filas con puesto que hoy
 704 cumplen: las partes no comparten más top 5 que hoy
  25 cumplen las seis

De las que cumplen las seis, cuántas tienen cada valor
  damping    0.15: 8 · 0.3: 10 · 0.5: 4 · 0.7: 3 · 0.85: 0
  inicio     por igual: 11 · por enunciados: 12 · solo cargas: 0 · solo derechos: 2
  lectura    lo recibido: 25 · con el reinicio: 0
  enlaces    los tres: 25 · solo modifies: 0 · solo depends_on: 0 · solo references: 0 · sin references: 0 · con términos: 0
  repetidos  cada vez: 12 · una vez: 13
  sin salida al inicio: 15 · a todas las filas: 10

Qué pruebas falla cada alternativa, con el resto como hoy
  damping: 0.15                 ninguna
  damping: 0.

## Lectura

Con los nueve grafos de `infra/json/kg` tal como están el 17 de noviembre de 2026:

- **Hoy.** 221 filas con puesto en las 18 lecturas y 43 empates. En Bellicum, Santander y
  OrvaneTalvera las dos partes comparten 10 de sus 15 primeras filas: 5 en Bellicum, 4 en
  Santander y 1 en OrvaneTalvera.
- **Damping.** No cambia qué filas tienen puesto ni los empates: las mismas 221 y 43 con los
  cinco valores. Entre 0.15 y 0.5 el orden es casi el mismo: con 0.15 y con 0.3, 25 y 27 de las
  30 primeras filas de hoy siguen entre las cinco primeras. Por encima de 0.5 pesa cada vez más
  lo que queda en las parejas mutuas, que crece con $d$ en los tres contratos (en Bellicum leído
  desde Bellicum Pharmaceuticals, del 64 % con 0.5 al 77 % con 0.85). En esa lectura, con 0.7
  §15.2 entra en las cinco primeras, y con 0.85 §9.1, §9.2, §15.2 y §15.1, a las que necesitan
  una o dos cláusulas, ocupan los puestos 2 a 5 por encima de §3.2, a la que necesitan diez. 0.5
  es el valor más alto de los probados sin ese defecto. Coincide con lo que
  `relevance_damping.ipynb` da para el contrato entero.
- **Dónde empieza.** Pesar el inicio por enunciados no cambia qué filas tienen puesto, deja 41
  empates en vez de 43, mueve 5 de las 30 primeras filas y no separa más a las partes (comparten
  las mismas 10). Empezar solo en las cargas deja sin inicio la lectura de Arequipa Todo
  Servicios, que no tiene ninguna, y quita el puesto a 12 filas. Empezar solo en los derechos se
  lo quita a 45: Santander pasa de 22 y 16 filas con puesto a 10 y 8.
- **Qué se lee.** Con el reinicio, 37 filas que alguna cláusula necesita quedan por debajo de
  filas que nadie necesita (6 y 7 en Bellicum, 4 y 4 en Santander, 5 y 6 en OrvaneTalvera), y el
  orden se parece más al tamaño de la fila (ρ medio de 0.20 a 0.31).
- **Enlaces.** Cada tipo solo deja menos de la mitad de las filas con puesto (104, 102 y 101 de
  221); solo `references` deja a Santander sin ninguna. Sin `references` se pierden 33. Con los
  términos definidos ganan puesto 29 filas, pero la primera pasa a ser la cláusula que define
  palabras, la misma para las dos partes: en Bellicum §1.1, que hoy es la 24.ª y la 21.ª, se
  lleva la mitad de todo lo recibido (50 % y 51 %; 37 % y 38 % contando una vez los repetidos);
  en Santander §5.8, por «IOF»; en OrvaneTalvera §13.1, por «Term». Ninguna de estas dos tiene
  puesto hoy. De las 30 primeras filas de hoy solo 14 siguen entre las cinco primeras.
- **Repetidos.** Contarlos una vez no cambia qué filas tienen puesto y sube los empates de 43 a
  52.
- **Sin salida.** Repartir el peso entre todas las filas da puesto a 13 filas más, y a ninguna
  la necesita una cláusula del camino de la parte. Las partes se parecen más: τ pasa de 0.89 a
  0.93 en Bellicum, de 0.76 a 0.97 en Santander y de 0.62 a 0.87 en OrvaneTalvera.
- **Combinaciones.** 25 de las 960 cumplen las seis pruebas, la de hoy entre ellas. Todas caminan
  los tres enlaces y leen lo recibido; ninguna usa $d = 0.85$ ni empieza solo en las cargas. La
  única ventaja sobre la de hoy son dos empates menos (41) con el inicio pesado por enunciados,
  y las que tienen más filas con puesto (234) son las que reparten el peso sin salida entre todas
  las filas.

**Decisión (17 de noviembre de 2026): se mantiene la configuración de hoy.**

| Decisión | Se queda en |
|---|---|
| Damping | 0.5 |
| Dónde empieza el paseo | En las cláusulas donde la parte tiene un enunciado, todas por igual |
| Qué se lee como relevancia | Lo que la fila recibe, sin el reinicio |
| Qué enlaces se caminan | `modifies`, `depends_on` y `references`, sin términos definidos |
| Enlaces repetidos | Cuentan cada vez |
| Cláusulas sin salida | Su peso vuelve al inicio |

Lo que este cuaderno no dice:

- **Que el orden sea el correcto.** Compara configuraciones sobre los mismos grafos; para saber
  si el orden acierta sigue faltando un criterio externo.
- **Que las pruebas sean las únicas posibles.** Las seis del apartado 9 son criterios de este
  experimento.
- **Nada sobre los contratos pequeños.** Los seis contratos con pocos enlaces casi no se mueven
  con ningún parámetro: las conclusiones salen de Bellicum, Santander y OrvaneTalvera. Y el
  adelantamiento de las parejas mutuas con $d$ alto solo llega a contarse en Bellicum; en los
  otros dos se ve que la parte que queda en ellas crece, pero no hay cláusulas que necesiten
  cinco o más por debajo.